In [25]:
"""
Automatically reloads libraries and utilities every time a cell is run
"""
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [26]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from utils.PSF_helpers import *
from utils.Plot_helpers import *
from utils.Zernike_helpers import *
from utils.Booth_helpers import *
from utils.Phase_diversity_helpers_cupy import *

In [27]:
N_order = 1              # 3 for 3P, 2 for 2P
lambd = 5.32e-4             # Wavelength [mm]
n = 1.333                  # Refractive index
k = 2*n*np.pi/lambd        # Wavenumber
num_apt = 1.2            # Numerical aperture
focal = 7.2                    # Focal length of objective (Olympus) [mm]
mag = np.nan                   # Magnification rate from input to objective
w_0 = np.nan                 # mm
alpha = np.arcsin(num_apt / n)

#106.5 microns
L_ffp = 0.1065
#512 x 512
grid_ffp = 512
grid = Centered_Square_Grid(L_ffp, grid_ffp, 0)
#CONJUGATE BFP LENGTH
grid_bfp = grid_ffp
L_bfp = (lambd * focal * grid_bfp) / L_ffp
microscope = Microscope(N_order, lambd, n, num_apt, focal, mag, w_0, L_bfp, grid_bfp)
x, y = grid.get_xy()
x_bfp = x * (L_bfp/ L_ffp)
y_bfp = y * (L_bfp/ L_ffp)

In [28]:
f = np.load("images/usaf_resolution.npy")

FIG S2(B)

In [ ]:
SNR_vals = [5, 10, 20]
gamma_vals = [10**(-x) for x in range(1, 31)]
num_samples = 1

modes_corrected = get_johnson_modes()

rng = np.random.default_rng(15)

bias_modes = [[-2,2]]
bias_strength = 500/523 
a_stack = [EmptyAberration()] + [Aberration([m], [bias_strength]) for m in bias_modes]

c_converged = np.zeros((num_samples, len(SNR_vals), len(gamma_vals), len(modes_corrected)))
c_true = np.zeros((num_samples, len(SNR_vals), len(gamma_vals), len(modes_corrected)))
for i in range(num_samples):

    true_aberration = generate_johnson_aberration(150/523, alpha, rng)
    img_stack = np.array([get_diversity_image(microscope, f, true_aberration, a) for a in a_stack])

    for j, SNR in enumerate(SNR_vals):
        d_stack = [add_gaussian_noise(img, SNR, rng) for img in img_stack]
        
        
        for k, gamma in enumerate(gamma_vals):

            params = {
                "gamma": gamma,
                "J_tol": 0.001,
            }


            c_guess, F_guess  = loop_optimize(n_loops = 100,
                                            microscope = microscope,
                                            d_stack = d_stack,
                                            a_stack = a_stack,
                                            modes_corrected = modes_corrected,
                                            params = params,
                                            debug = False,
                                            log = False)
            
            c_converged[i, j, k] = c_guess
            c_true[i, j, k] = true_aberration.strengths

np.save("results/gamma_1p_true.npy", c_true)
np.save("results/gamma_1p_converged.npy", c_converged)

FIG S2(C)

In [ ]:
SNR_vals = [3.0, 5.0, 10.0, 20.0]
rms_vals = [0.02, 0.04, 0.06, 0.08, 0.12, 0.14, 0.16, 0.18, 0.20, 0.22, 0.24]
num_samples = 1

modes_corrected = get_johnson_modes()
rng = np.random.default_rng(15)

GAMMA = 1e-6

bias_modes = [[2,2],
               [-2,2],
               [2,2],
               [-2,2]]



#convert from microns to waves -> 1000 nm per micron, 1 wv per 523 nm
bias_strengths = np.array([0.5, 0.5, -0.5, -0.5], dtype = np.float64) * (1000/523)
a_stack = [EmptyAberration()] + [Aberration([bias_modes[i]], [bias_strengths[i]]) for i in range(len(bias_modes))]

c_converged = np.zeros((num_samples, len(SNR_vals), len(rms_vals), len(modes_corrected)))
c_true = np.zeros((num_samples, len(SNR_vals), len(rms_vals), len(modes_corrected)))
for i in range(num_samples):

    raw_aberration = generate_johnson_aberration(1, alpha, rng)
    
    for j, SNR in enumerate(SNR_vals):
        for k, rms in enumerate(rms_vals):

            #rescale the strength of the true aberration
            true_aberration = rescale_aberration(raw_aberration, rms, alpha)
            img_stack = np.array([get_diversity_image(microscope, f, true_aberration, a) for a in a_stack])
            d_stack = [add_gaussian_noise(img, SNR, rng) for img in img_stack]
            
            params = {
                "gamma": GAMMA,
                "J_tol": 0.001,
            }

            c_guess, F_guess  = loop_optimize(n_loops = 100,
                                            microscope = microscope,
                                            d_stack = d_stack,
                                            a_stack = a_stack,
                                            modes_corrected = modes_corrected,
                                            params = params,
                                            debug = False,
                                            log = False)
            
            c_converged[i, j, k] = c_guess
            c_true[i, j, k] = true_aberration.strengths

np.save("results/gamma_1p_true.npy", c_true)
np.save("results/gamma_1p_converged.npy", c_converged)



  8%|▊         | 8/100 [00:05<00:58,  1.57it/s]


KeyboardInterrupt: 